In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark session
spark = SparkSession.builder \
    .appName("CS131_Sprint5_DataCleaning_ENG1") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("✅ Spark session initialized!")

# Load raw customer data
raw_df = spark.read.csv("../data/customer_data.csv", header=True, inferSchema=True)

print(f"\n{'='*50}")
print("RAW DATA LOADED")
print(f"{'='*50}")
print(f"Total rows: {raw_df.count()}")
print(f"Total columns: {len(raw_df.columns)}")
print(f"\nColumns: {raw_df.columns}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/09 12:03:36 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.111 instead (on interface en0)
25/11/09 12:03:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/09 12:03:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/09 12:03:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Spark session initialized!

RAW DATA LOADED
Total rows: 100000
Total columns: 12

Columns: ['id', 'age', 'gender', 'income', 'education', 'region', 'loyalty_status', 'purchase_frequency', 'purchase_amount', 'product_category', 'promotion_usage', 'satisfaction_score']


In [2]:
print(f"{'='*50}")
print("DATA QUALITY ANALYSIS")
print(f"{'='*50}")

# Show sample data
print("\nSample of raw data:")
raw_df.show(5, truncate=False)

# Check for NULL values
print("\nNULL counts by column:")
null_counts = raw_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in raw_df.columns
])
null_counts.show()

# Check for whitespace issues in string columns
print("\nChecking for leading/trailing whitespace in string columns:")
string_columns = [field.name for field in raw_df.schema.fields 
                  if str(field.dataType) == "StringType"]
print(f"String columns: {string_columns}")

# Show a sample with potential whitespace
for col_name in string_columns[:3]:  # Check first 3 string columns
    print(f"\nSample values from '{col_name}':")
    raw_df.select(col_name).distinct().show(5, truncate=False)

DATA QUALITY ANALYSIS

Sample of raw data:
+---+---+------+------+----------+------+--------------+------------------+---------------+----------------+---------------+------------------+
|id |age|gender|income|education |region|loyalty_status|purchase_frequency|purchase_amount|product_category|promotion_usage|satisfaction_score|
+---+---+------+------+----------+------+--------------+------------------+---------------+----------------+---------------+------------------+
|1  |27 |Male  |40682 |Bachelor  |East  |Gold          |frequent          |18249          |Books           |0              |6                 |
|2  |29 |Male  |15317 |Masters   |West  |Regular       |rare              |4557           |Clothing        |1              |6                 |
|3  |37 |Male  |38849 |Bachelor  |West  |Silver        |rare              |11822          |Clothing        |0              |6                 |
|4  |30 |Male  |11568 |HighSchool|South |Regular       |frequent          |4098           |Fo

In [3]:
print(f"{'='*50}")
print("STEP 1: TRIMMING WHITESPACE")
print(f"{'='*50}")

# Get all string columns
string_columns = [field.name for field in raw_df.schema.fields 
                  if str(field.dataType) == "StringType"]

print(f"Trimming {len(string_columns)} string columns: {string_columns}")

# Trim whitespace from all string columns
cleaned_df = raw_df
for col_name in string_columns:
    cleaned_df = cleaned_df.withColumn(col_name, F.trim(F.col(col_name)))

print("✅ Whitespace trimming complete!")

# Show before/after comparison for one column
if string_columns:
    sample_col = string_columns[0]
    print(f"\nBefore/After comparison for '{sample_col}':")
    print("Before:")
    raw_df.select(sample_col).show(3, truncate=False)
    print("After:")
    cleaned_df.select(sample_col).show(3, truncate=False)

STEP 1: TRIMMING WHITESPACE
Trimming 0 string columns: []
✅ Whitespace trimming complete!


In [4]:
print(f"{'='*50}")
print("STEP 2: NORMALIZING STRING DATA")
print(f"{'='*50}")

# Normalize specific columns (convert to lowercase/standardize)
# Adjust these based on your actual columns

# Example: Normalize gender, education, region, loyalty_status, purchase_frequency, product_category
columns_to_normalize = ['gender', 'education', 'region', 'loyalty_status', 
                        'purchase_frequency', 'product_category']

for col_name in columns_to_normalize:
    if col_name in cleaned_df.columns:
        print(f"Normalizing '{col_name}' to lowercase...")
        cleaned_df = cleaned_df.withColumn(col_name, F.lower(F.col(col_name)))

print("✅ Normalization complete!")

# Show normalized data sample
print("\nNormalized data sample:")
cleaned_df.select(columns_to_normalize).show(5, truncate=False)

STEP 2: NORMALIZING STRING DATA
Normalizing 'gender' to lowercase...
Normalizing 'education' to lowercase...
Normalizing 'region' to lowercase...
Normalizing 'loyalty_status' to lowercase...
Normalizing 'purchase_frequency' to lowercase...
Normalizing 'product_category' to lowercase...
✅ Normalization complete!

Normalized data sample:
+------+----------+------+--------------+------------------+----------------+
|gender|education |region|loyalty_status|purchase_frequency|product_category|
+------+----------+------+--------------+------------------+----------------+
|male  |bachelor  |east  |gold          |frequent          |books           |
|male  |masters   |west  |regular       |rare              |clothing        |
|male  |bachelor  |west  |silver        |rare              |clothing        |
|male  |highschool|south |regular       |frequent          |food            |
|female|college   |north |regular       |occasional        |clothing        |
+------+----------+------+------------

In [5]:
print(f"{'='*50}")
print("STEP 3: HANDLING NULL VALUES")
print(f"{'='*50}")

# Show NULL counts BEFORE handling
print("NULL counts BEFORE cleaning:")
null_counts_before = cleaned_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in cleaned_df.columns
])
null_counts_before.show()

rows_before = cleaned_df.count()

# Strategy: Drop rows with NULLs in critical columns
# For customer behavior analysis, we need complete records for key fields
critical_columns = ['id', 'age', 'gender', 'income']

print(f"\nDropping rows with NULL values in critical columns: {critical_columns}")
cleaned_df = cleaned_df.dropna(subset=critical_columns)

# For non-critical columns, fill with defaults
# Example: Fill missing loyalty_status with "unknown"
if 'loyalty_status' in cleaned_df.columns:
    cleaned_df = cleaned_df.fillna({'loyalty_status': 'unknown'})

if 'purchase_frequency' in cleaned_df.columns:
    cleaned_df = cleaned_df.fillna({'purchase_frequency': 'unknown'})

rows_after = cleaned_df.count()

# Show NULL counts AFTER handling
print("\nNULL counts AFTER cleaning:")
null_counts_after = cleaned_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in cleaned_df.columns
])
null_counts_after.show()

print(f"\n✅ NULL handling complete!")
print(f"Rows before: {rows_before}")
print(f"Rows after: {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")

STEP 3: HANDLING NULL VALUES
NULL counts BEFORE cleaning:
+---+---+------+------+---------+------+--------------+------------------+---------------+----------------+---------------+------------------+
| id|age|gender|income|education|region|loyalty_status|purchase_frequency|purchase_amount|product_category|promotion_usage|satisfaction_score|
+---+---+------+------+---------+------+--------------+------------------+---------------+----------------+---------------+------------------+
|  0|  0|     0|     0|        0|     0|             0|                 0|              0|               0|              0|                 0|
+---+---+------+------+---------+------+--------------+------------------+---------------+----------------+---------------+------------------+


Dropping rows with NULL values in critical columns: ['id', 'age', 'gender', 'income']

NULL counts AFTER cleaning:
+---+---+------+------+---------+------+--------------+------------------+---------------+----------------+---

In [6]:
print(f"{'='*50}")
print("CLEANING SUMMARY")
print(f"{'='*50}")

print("\nBEFORE CLEANING:")
print(f"  Total rows: {raw_df.count()}")
print(f"  Total columns: {len(raw_df.columns)}")

print("\nAFTER CLEANING:")
print(f"  Total rows: {cleaned_df.count()}")
print(f"  Total columns: {len(cleaned_df.columns)}")
print(f"  Data retention: {(cleaned_df.count() / raw_df.count() * 100):.2f}%")

print("\nCleaned data sample:")
cleaned_df.show(10, truncate=False)

print("\nCleaned data schema:")
cleaned_df.printSchema()

CLEANING SUMMARY

BEFORE CLEANING:
  Total rows: 100000
  Total columns: 12

AFTER CLEANING:
  Total rows: 100000
  Total columns: 12
  Data retention: 100.00%

Cleaned data sample:
+---+---+------+------+----------+------+--------------+------------------+---------------+----------------+---------------+------------------+
|id |age|gender|income|education |region|loyalty_status|purchase_frequency|purchase_amount|product_category|promotion_usage|satisfaction_score|
+---+---+------+------+----------+------+--------------+------------------+---------------+----------------+---------------+------------------+
|1  |27 |male  |40682 |bachelor  |east  |gold          |frequent          |18249          |books           |0              |6                 |
|2  |29 |male  |15317 |masters   |west  |regular       |rare              |4557           |clothing        |1              |6                 |
|3  |37 |male  |38849 |bachelor  |west  |silver        |rare              |11822          |clothin

In [8]:
print(f"{'='*50}")
print("SAVING CLEANED DATA")
print(f"{'='*50}")

# Save as Parquet (efficient format for Spark)
output_path = "../sprint5_outputs/cleaned_customer_data.parquet"

print(f"Saving cleaned data to: {output_path}")
cleaned_df.write.mode("overwrite").parquet(output_path)

print("✅ Cleaned data saved!")

# Verify saved data
print("\nVerifying saved data...")
verify_df = spark.read.parquet(output_path)
print(f"Verified rows: {verify_df.count()}")
print(f"Verified columns: {len(verify_df.columns)}")

print("\n🎉 DATA CLEANING PIPELINE COMPLETE!")
print(f"\nCleaned data is ready at: {output_path}")

SAVING CLEANED DATA
Saving cleaned data to: ../sprint5_outputs/cleaned_customer_data.parquet
✅ Cleaned data saved!

Verifying saved data...
Verified rows: 100000
Verified columns: 12

🎉 DATA CLEANING PIPELINE COMPLETE!

Cleaned data is ready at: ../sprint5_outputs/cleaned_customer_data.parquet
